## Imports and Paths

In [20]:
import json
import subprocess
from datetime import datetime
import h5py
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime

In [18]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

SOLAR_DIR = RAW_DIR / "solar_observations"
TEC_HDF5_DIR = RAW_DIR / "tec_hdf5"
TEC_GRID_DIR = PROCESSED_DIR / "tec_grids"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
RESULTS_DIR = OUTPUT_DIR / "results"

FLARE_MANIFEST = DATA_DIR / "flare_manifest.csv"

## Data Extraction & Preparation

In [7]:
with open(Path(RESULTS_DIR) / "query_result.json", "r") as f:
    data = json.load(f)


In [9]:
records = data['records']

m_flares = [
    flare for flare in records if flare["flare_class"].startswith('M')
]

x_flares = [
    flare for flare in records if flare["flare_class"].startswith('X')
]
selected_flares = m_flares[:15] + x_flares[:15]

In [10]:
print(f"M-class available: {len(m_flares)}")
print(f"X-class available: {len(x_flares)}")
print(f"Selected: {len(selected_flares)}")

M-class available: 94
X-class available: 57
Selected: 30


In [11]:
dates = sorted({
    datetime.strptime(
        flare["peak_time"],
        "%Y-%m-%d %H:%M:%S"
    ).strftime("%m/%d/%Y")
    for flare in selected_flares
})

In [ ]:
# for date in dates:
#     cmd = [
#         "globalDownload.py",
#         "--verbose",
#         "--url=https://cedar.openmadrigal.org/",
#         "--outputDir=./tec_hdf5",
#         '--user_fullname=Amrit+Madabushi',
#         "--user_email=am2699@njit.edu",
#         '--user_affiliation=NJIT',
#         '--format=hdf5',
#         f"--startDate={date}",
#         f"--endDate={date}",
#         "--inst=8000",
#         "--kindat=3500"
#     ]
#     print(f"Downloading TEC for {date}")
#     subprocess.run(cmd, check=True)

In [13]:
CHUNK_SIZE = 250_000

def get_tec_file(flare):
    """Find the daily TEC file corresponding to a flare's peak date."""
    peak = pd.to_datetime(flare["peak_time"])
    date_code = peak.strftime("%y%m%d")

    # print(date_code)
    matches = sorted(TEC_HDF5_DIR.glob(f"gps{date_code}g.*.hdf5"))

    if not matches:
        raise FileNotFoundError(
            f"No TEC file found for {peak.date()}"
        )
    return matches[0]

def extract_nearest_tec_map(target_time):
    """
    Find the TEC snapshot nearest to target_time wtihout loading the entire daily HDF5 into memory.
    """
    target_time = pd.to_datetime(target_time)

    #Select HDF5 file based on target_date
    date_code = target_time.strftime("%y%m%d")
    matches = sorted(TEC_HDF5_DIR.glob(f"gps{date_code}g.*.hdf5"))
    if not matches:
        raise FileNotFoundError(
            f"No TEC file found for {target_time.date()}"
        )
    path = matches[0]

    nearest_time = None
    nearest_diff = None

    # PASS 1: find nearest avaliable TEC timestamp
    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0,len(table),CHUNK_SIZE):
            chunk = table[start:start+CHUNK_SIZE]

            times = pd.to_datetime({
                "year": chunk["year"],
                "month": chunk["month"],
                "day": chunk["day"],
                "hour": chunk["hour"],
                "minute": chunk["min"],
                "second": chunk["sec"],
            })

            unique_times = pd.Index(times.unique())

            diffs = abs(unique_times-target_time)
            i = diffs.argmin()

            candidate_time = unique_times[i]
            candidate_diff = diffs[i]

            if nearest_diff is None or candidate_diff < nearest_diff:
                nearest_diff = candidate_diff
                nearest_time = candidate_time
        print(f"Requested: {target_time}")
    print(f"Nearest TEC map: {nearest_time}")
    print(f"Difference: {nearest_diff}")
    if nearest_diff > pd.Timedelta(minutes=5):
        raise ValueError(
            f"TEC timestamp mismatch: requested {target_time}, nearest {nearest_time}, difference {nearest_diff}"
        )

    # PASS 2: extract only rows belonging to that TEC timestamp
    pieces = []

    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0, len(table), CHUNK_SIZE):
            chunk = table[start:start + CHUNK_SIZE]

            mask = (
                (chunk["year"] == nearest_time.year) &
                (chunk["month"] == nearest_time.month) &
                (chunk["day"] == nearest_time.day) &
                (chunk["hour"] == nearest_time.hour) &
                (chunk["min"] == nearest_time.minute) &
                (chunk["sec"] == nearest_time.second)
            )

            if np.any(mask):
                selected = chunk[mask]

                pieces.append(
                    pd.DataFrame({
                        "gdlat": selected["gdlat"],
                        "glon": selected["glon"],
                        "tec": selected["tec"],
                    })
                )

    if not pieces:
        raise RuntimeError(
            f"No TEC rows found for {nearest_time}"
        )

    tec_map = pd.concat(pieces, ignore_index=True)

    # Madrigal missing-value convention
    tec_map.loc[tec_map["tec"] > 1e20, "tec"] = np.nan

    return nearest_time, tec_map

In [14]:
flare = selected_flares[0]

path = get_tec_file(flare)

print("Flare:", flare["flare_class"], flare["peak_time"])
print("TEC file:", path.name)

tec_time, tec_map = extract_nearest_tec_map(
    flare["peak_time"]
)

print(tec_map.shape)
print(tec_map.head())

Flare: M6.6 2011-02-13 17:38:00
TEC file: gps110213g.002.hdf5
Requested: 2011-02-13 17:38:00
Nearest TEC map: 2011-02-13 17:37:30
Difference: 0 days 00:00:30
(6687, 3)
   gdlat  glon   tec
0  -90.0 -84.0  66.0
1  -90.0 -82.0  65.9
2  -90.0 -81.0  65.9
3  -90.0 -80.0  65.8
4  -90.0 -79.0  65.7


In [15]:
from pathlib import Path

print("cwd:", Path.cwd())
print("TEC_DIR:", TEC_HDF5_DIR.resolve())
print("exists:", TEC_HDF5_DIR.exists())
print("files found:", list(TEC_HDF5_DIR.glob("gps110213g*.hdf5")))

cwd: /home/amritm/projects/surya-tec-forecasting/notebooks
TEC_DIR: /home/amritm/projects/surya-tec-forecasting/data/raw/tec_hdf5
exists: True
files found: [PosixPath('/home/amritm/projects/surya-tec-forecasting/data/raw/tec_hdf5/gps110213g.002.hdf5')]


### Covering missing dates

In [ ]:
# required_dates = {
#     (pd.to_datetime(flare["peak_time"]) + pd.Timedelta(hours = offset)).date()
#     for flare in selected_flares
#     for offset in range(-4,4)
# }

# available_dates = {
#     pd.to_datetime(path.name[3:9], format="%y%m%d").date()
#     for path in TEC_DIR.glob("gps*g.*.hdf5")
# }

# missing_dates = sorted(required_dates-available_dates)

# print("Required dates:", len(required_dates))
# print("Available dates:", len(available_dates))
# print("Missing dates:", missing_dates)

In [ ]:
# dates = [
#     date.strftime("%m/%d/%Y")
#     for date in missing_dates
# ]

# for date in dates:
#     cmd = [
#         "globalDownload.py",
#         "--verbose",
#         "--url=https://cedar.openmadrigal.org/",
#         "--outputDir=./tec_hdf5",
#         "--user_fullname=Amrit+Madabushi",
#         "--user_email=am2699@njit.edu",
#         "--user_affiliation=NJIT",
#         "--format=hdf5",
#         f"--startDate={date}",
#         f"--endDate={date}",
#         "--inst=8000",
#         "--kindat=3500"
#     ]

#     print(f"Downloading TEC for {date}")
#     subprocess.run(cmd, check=True)

In [ ]:
# from pathlib import Path
# import pandas as pd


# extra_files = []

# for path in TEC_DIR.glob("*.hdf5"):
#     file_date = pd.to_datetime(
#         path.name[3:9], format="%y%m%d"
#     ).date()

#     if file_date not in required_dates or "_0.hdf5" in path.name:
#         extra_files.append(path)

# print("Extra files:")
# for path in extra_files:
#     print(path.name)

### Visualization

In [ ]:
def plot_map(tec_map, timestamp):
    """Plot and save a global TEC map for the given timestamp."""
    grid = tec_map.pivot(
        index="gdlat",
        columns="glon",
        values="tec"
    )

    fig,ax = plt.subplots(figsize=(12,5))

    mesh = ax.pcolormesh(
        grid.columns,
        grid.index,
        grid.values,
        shading="auto"
    )

    fig.colorbar(mesh,ax=ax,label="TEC")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Global TEC Map —  {timestamp}")

    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)

    timestamp = pd.to_datetime(timestamp)
    filename = f"tec_{timestamp.strftime('%Y%m%d_%H%M%S')}.png"
    save_path = FIGURE_DIR / filename

    fig.savefig(save_path, dpi=300, bbox_inches="tight")

    print(f"Saved: {save_path}")

    # plt.show()
    plt.close(fig)

In [ ]:
# plot_map(tec_map,tec_time)

In [ ]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])

#     for offset in [-3,-2, -1, 0, 1, 2, 3]:
#         target_time = peak + pd.Timedelta(hours=offset)

#         tec_time, tec_map = extract_nearest_tec_map(target_time)

#         plot_map(tec_map, tec_time)

In [ ]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])

#     target_time = peak + pd.Timedelta(hours=-4)

#     tec_time, tec_map = extract_nearest_tec_map(target_time)

#     plot_map(tec_map, tec_time)

### Obtain T-4 solar observations from Amazon S3 Nasa Surya Bench

In [ ]:
import boto3
from botocore import UNSIGNED

In [ ]:
config = boto3.session.Config(signature_version=UNSIGNED)
s3 = boto3.client("s3", config=config)
bucket = 'nasa-surya-bench'

In [ ]:
# response = s3.list_objects_v2(Bucket=bucket)
# for obj in response.get('Contents', []):
#     print(obj['Key'])

In [ ]:
dates

['01/07/2014',
 '01/23/2012',
 '02/13/2011',
 '02/15/2011',
 '03/07/2012',
 '03/09/2011',
 '03/09/2012',
 '03/10/2012',
 '03/29/2014',
 '04/11/2013',
 '04/25/2014',
 '05/10/2012',
 '07/02/2012',
 '07/04/2012',
 '07/05/2012',
 '07/12/2012',
 '07/30/2011',
 '08/03/2011',
 '08/04/2011',
 '09/06/2011',
 '09/07/2011',
 '09/08/2011',
 '09/10/2014',
 '09/25/2011',
 '10/22/2014',
 '10/24/2014',
 '11/08/2013',
 '11/10/2013']

In [ ]:


# for date_str in dates:
#     date = datetime.strptime(date_str, "%m/%d/%Y")
#     prefix = date.strftime("%Y/%m/%Y%m%d_")

#     response = s3.list_objects_v2(
#         Bucket=bucket,
#         Prefix=prefix
#     )

#     print(date_str, len(response.get("Contents", [])))

In [ ]:
# for flare in selected_flares:
#     print(flare["peak_time"])

In [ ]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak-pd.Timedelta(hours=4)
#     print(f"Peak: {peak} Cutoff: {cutoff}")

In [ ]:
# responses = []
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak-pd.Timedelta(hours=4)
#     prefix = cutoff.strftime("%Y/%m/%Y%m%d_")
#     response = s3.list_objects_v2(
#         Bucket=bucket,
#         Prefix=prefix
#     )

#     keys = [obj["Key"] for obj in response.get("Contents", [])]

#     valid_keys = [
#         key for key in keys
#         if pd.to_datetime(key.split("/")[-1], format="%Y%m%d_%H%M.nc") <= cutoff
#     ]

#     if not valid_keys:
#         print(f"No valid observation found for {peak}")
#         continue

#     selected_key = max(valid_keys)

#     responses.append((peak,cutoff,selected_key))

#     print(date_str, len(response.get("Contents", [])))
#     print(f"Peak: {peak} | Cutoff: {cutoff} | S3: {selected_key}")

Note: September 10, 2014 measurement has a temporal difference of 45 mins between the selected flare observation and cutoff

In [ ]:
# from pathlib import Path

# output_dir = Path("solar_observations")
# output_dir.mkdir(exist_ok=True)

In [ ]:
# for peak,cutoff,key in responses:
#     destination = output_dir / Path(key).name

#     if not destination.exists(): # avoid redownloading
#         s3.download_file(bucket, key, str(destination))

#     print(f"Downloaded: {destination}")

In [ ]:
# from pathlib import Path

# solar_files = sorted(Path("solar_observations").glob("*.nc"))

# print(f"Total solar files: {len(solar_files)}")

# for file in solar_files:
#     print(file.name)

In [ ]:
# for flare in selected_flares:    
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak-pd.Timedelta(hours=5)
#     prefix = cutoff.strftime("%Y/%m/%Y%m%d_")
#     response = s3.list_objects_v2(
#         Bucket=bucket,
#         Prefix=prefix
#     )

#     keys = [obj["Key"] for obj in response.get("Contents", [])]

#     valid_keys = [
#         key for key in keys
#         if pd.to_datetime(key.split("/")[-1], format="%Y%m%d_%H%M.nc") <= cutoff
#     ]

#     if not valid_keys:
#         print(f"No valid observation found for {peak}")
#         continue

#     selected_key = max(valid_keys)
#     destination = output_dir / Path(selected_key).name
#     if not destination.exists():
#         s3.download_file(bucket, selected_key, str(destination))
#         print(f"Downloaded: {destination.name}")


In [ ]:
# from pathlib import Path

# solar_files = list(Path("solar_observations").glob("*.nc"))

# print(f"Total files: {len(solar_files)}")
# print(f"Total size: {sum(f.stat().st_size for f in solar_files) / 1024**3:.2f} GiB")

In [ ]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak - pd.Timedelta(hours=5)

#     candidates = []

#     for path in output_dir.glob("*.nc"):
#         observation_time = pd.to_datetime(
#             path.stem,
#             format="%Y%m%d_%H%M"
#         )

#         if observation_time <= cutoff:
#             candidates.append((observation_time, path.name))

#     if candidates:
#         observation_time, filename = max(candidates)
#         difference = (cutoff - observation_time).total_seconds() / 60

#         print(
#             f"Peak: {peak} | "
#             f"T-5 Cutoff: {cutoff} | "
#             f"Selected: {observation_time} | "
#             f"Difference: {difference:.0f} min | "
#             f"File: {filename}"
#         )

In [ ]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff_t4 = peak - pd.Timedelta(hours=4)
#     cutoff_t5 = peak - pd.Timedelta(hours=5)

#     candidates_t4 = []
#     candidates_t5 = []

#     for path in output_dir.glob("*.nc"):
#         observation_time = pd.to_datetime(path.stem, format="%Y%m%d_%H%M")

#         if observation_time <= cutoff_t4:
#             candidates_t4.append((observation_time, path.name))

#         if observation_time <= cutoff_t5:
#             candidates_t5.append((observation_time, path.name))

#     if candidates_t4 and candidates_t5:
#         t4, file_t4 = max(candidates_t4)
#         t5, file_t5 = max(candidates_t5)

#         gap = (t4 - t5).total_seconds() / 60

#         print(
#             f"Peak: {peak} | "
#             f"T-5: {t5} | "
#             f"T-4: {t4} | "
#             f"Gap: {gap:.0f} min"
#         )

The gap between $t-5$ and $t-4$ for the September 10, 2014 flare is only 24 minutes, consistent with the previously identified 45-minute offset at $t-4$. The March 9th, 2011 flare a gap of 48 minutes. All other solar observations have a 60 minute gap between $t-4$ and $t-5$

## Preprocessing

In [ ]:
tec_files = list(Path(FIGURE_DIR/"tec_maps").glob("tec_*.png"))

In [ ]:
# flare_df = pd.DataFrame(selected_flares)
# for offset in [-5, -4]:
#     flare_df[f"solar_t{offset}"] = None

# for offset in range(-3, 4):
#     flare_df[f"tec_t{offset:+d}"] = None

# solar_times = {
#     path: pd.to_datetime(path.stem,format="%Y%m%d_%H%M")
#     for path in solar_files
# }

# tec_times = {
#     path: pd.to_datetime(
#         path.stem.removeprefix("tec_"),
#         format="%Y%m%d_%H%M%S"
#     )
#     for path in tec_files
# }
# hdf5_files = list(Path("tec_hdf5").glob("*.hdf5"))
# hdf5_by_date = {
#     pd.to_datetime(path.name[3:9], format="%y%m%d").date(): path
#     for path in hdf5_files
# }

In [ ]:
# with h5py.File("tec_hdf5/gps110213g.002.hdf5", "r") as f:
#     dataset = f["Data"]["Table Layout"]

#     print(dataset.shape)
#     print(dataset.dtype.names)

In [ ]:
# for idx, flare in flare_df.iterrows():
#     peak_time = pd.Timestamp(flare["peak_time"])

#     # Match existing solar observations to solar_t5 and solar_t4
#     for offset in [-5,-4]:
#         target_time = peak_time + pd.Timedelta(hours=offset)
        
#         closest_file = min(
#             solar_times,
#             key=lambda path: abs(solar_times[path] - target_time)
#         )

#         flare_df.at[idx, f"solar_t{offset:+d}"] = str(closest_file)
            
#     # Match existing TEC maps to the seven target timestamps
#     for offset in range(-3, 4):
#         col = f"tec_t{offset:+d}"
#         target_time = peak_time + pd.Timedelta(hours=offset)

#         closest_file = min(
#             tec_times,
#             key=lambda path: abs(tec_times[path] - target_time)
#         )

#         flare_df.at[idx, col] = str(closest_file)

#         tec_time = tec_times[closest_file]

#         flare_df.at[idx, f"{col}_hdf5"] = str(
#             hdf5_by_date.get(tec_time.date(), "")
#         )
            
# flare_df.to_csv("flare_manifest.csv", index=False)

In [ ]:
def grid_tec(tec_map, resolution=1.0):
    '''
    This function is used to create a grid based on latitude, longitude and total electron content (TEC) measurements 
    provided by the map.
    
    Longitude coordinates are normalized to [-180, 180], and spatial binning is applied to latitude and longitude. 
    TEC measurements within each grid cell are averaged, with cells containing no valid observations reported as NaN. 
    '''
    lat = tec_map["gdlat"].to_numpy()
    lon = tec_map["glon"].to_numpy()
    tec = tec_map["tec"].to_numpy()

    # Normalize Longitude
    lon = ((lon+180)%360)-180

    # Remove invalid measurements
    valid = (np.isfinite(lat) & np.isfinite(lon) & np.isfinite(tec))

    lat = lat[valid]
    lon = lon[valid]
    tec = tec[valid]

    # Grid boundaries
    if resolution <= 0 or not np.isclose(180 / resolution, round(180 / resolution)):
        raise ValueError("Resolution must evenly divide 180 degrees.")
    lat_edges = np.linspace(-90,90,int(180/resolution)+1)
    lon_edges = np.linspace(-180,180,int(360/resolution)+1)

    # Aggregate TEC values within each grid cell
    tec_sum, _, _ = np.histogram2d(
        lat, lon,
        bins=[lat_edges, lon_edges],
        weights=tec
    )

    # Count meaasurements within each cell
    counts, _, _ = np.histogram2d(
        lat, lon,
        bins=[lat_edges, lon_edges]
    )

    # Calculate mean TEC per cell
    grid = np.full(tec_sum.shape, np.nan, dtype=np.float32)

    np.divide(tec_sum,counts,out=grid,where=counts>0)

    return grid

In [ ]:
def extract_nearest_tec_map(target_time, path):
    """
    Extract the TEC map from the hdf5 file
    """
    CHUNK_SIZE = 250_000
    
    target_time = pd.to_datetime(target_time)


    nearest_time = None
    nearest_diff = None

    # PASS 1: find nearest avaliable TEC timestamp
    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0,len(table),CHUNK_SIZE):
            chunk = table[start:start+CHUNK_SIZE]

            times = pd.to_datetime({
                "year": chunk["year"],
                "month": chunk["month"],
                "day": chunk["day"],
                "hour": chunk["hour"],
                "minute": chunk["min"],
                "second": chunk["sec"],
            })

            unique_times = pd.Index(times.unique())

            diffs = abs(unique_times-target_time)
            i = diffs.argmin()

            candidate_time = unique_times[i]
            candidate_diff = diffs[i]

            if nearest_diff is None or candidate_diff < nearest_diff:
                nearest_diff = candidate_diff
                nearest_time = candidate_time
        print(f"Requested: {target_time}")
    print(f"Nearest TEC map: {nearest_time}")
    print(f"Difference: {nearest_diff}")
    if nearest_diff > pd.Timedelta(minutes=5):
        raise ValueError(
            f"TEC timestamp mismatch: requested {target_time}, nearest {nearest_time}, difference {nearest_diff}"
        )

    # PASS 2: extract only rows belonging to that TEC timestamp
    pieces = []

    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0, len(table), CHUNK_SIZE):
            chunk = table[start:start + CHUNK_SIZE]

            mask = (
                (chunk["year"] == nearest_time.year) &
                (chunk["month"] == nearest_time.month) &
                (chunk["day"] == nearest_time.day) &
                (chunk["hour"] == nearest_time.hour) &
                (chunk["min"] == nearest_time.minute) &
                (chunk["sec"] == nearest_time.second)
            )

            if np.any(mask):
                selected = chunk[mask]

                pieces.append(
                    pd.DataFrame({
                        "gdlat": selected["gdlat"],
                        "glon": selected["glon"],
                        "tec": selected["tec"],
                    })
                )

    if not pieces:
        raise RuntimeError(
            f"No TEC rows found for {nearest_time}"
        )

    tec_map = pd.concat(pieces, ignore_index=True)

    # Madrigal missing-value convention
    tec_map.loc[tec_map["tec"] > 1e20, "tec"] = np.nan

    return nearest_time, tec_map

In [ ]:
# Used to generate the precomputed TEC Grids stored in npz files instead of accessing from hdf5 files
# from pathlib import Path

# output_dir = Path("tec_grids")
# output_dir.mkdir(exist_ok=True)

# for idx,flare in dataset.df.iterrows():
#     tec, tec_mask = dataset.load_tec(flare)

#     output_path = output_dir / f"flare_{idx:03d}.npz"

#     np.savez_compressed(
#         output_path,
#         tec=tec,
#         tec_mask = tec_mask,
#         resolution=1.0
#     )
#     dataset.df.loc[idx, "tec_grid_path"] = str(output_path)


# dataset.df.to_csv("flare_manifest.csv", index=False)

In [ ]:
# # Updating the paths as per the repository structure


# manifest_path = DATA_DIR / "flare_manifest.csv"
# df = pd.read_csv(manifest_path)

# replacements = {
#     "solar_observations/": "data/raw/solar_observations/",
#     "figures/tec_maps/": "outputs/figures/tec_maps/",
#     "tec_hdf5/": "data/raw/tec_hdf5/",
#     "tec_grids/": "data/processed/tec_grids/",
# }

# for col in df.select_dtypes(include="object").columns:
#     for old, new in replacements.items():
#         df[col] = df[col].str.replace(
#             old, new, regex=False
#         )

# df.to_csv(manifest_path, index=False)

/tmp/ipykernel_12761/3737338147.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [ ]:
# target_time = pd.Timestamp("2011-02-13 14:38:00")

# tec_time, tec_map = extract_nearest_tec_map(target_time)

# tec_grid = grid_tec(tec_map)

# print("Timestamp:", tec_time)
# print("Grid shape:", tec_grid.shape)
# print("Valid cells:", np.isfinite(tec_grid).sum())
# print("Missing cells:", np.isnan(tec_grid).sum())

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12, 6))

# plt.imshow(
#     tec_grid,
#     origin="lower",
#     extent=[-180, 180, -90, 90],
#     aspect="auto",
#     cmap="viridis"
# )

# plt.colorbar(label="TEC (TECU)")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.title(f"Gridded TEC — {tec_time}")

# plt.show()

In [ ]:
# for resolution in [0.5, 1.0, 2.0]:
#     grid = grid_tec(tec_map, resolution=resolution)

#     coverage = np.isfinite(grid).mean() * 100

#     print(
#         f"Resolution: {resolution}° | "
#         f"Shape: {grid.shape} | "
#         f"Coverage: {coverage:.2f}%"
#     )

For this experiment, we will be using 1.0° resolution.